---
title: "Exercise 8: scDRS Results and Interpretation"
subtitle: "Post-GWAS Analysis Course"
format:
  html:
    embed-resources: true
    toc: true
    toc-depth: 3
execute:
  message: false
  warning: false
jupyter: python
---

# Overview

This notebook presents and interprets the scDRS results from the previous notebook. You will visualise cell-level scores, inspect the group-level associations, and check how robust the conclusions are to alternative gene sets.

::: {.callout-note}
## Learning goals
By the end of this notebook, you should be able to:
- present scDRS scores and group-level associations in a clear visual form
- explain what the main enrichment outputs mean biologically
- discuss basic sensitivity checks for cell-type association results
:::

::: callout-warning
You will need the following input files:
- scDRS scores: {trait}.full_scores.gz
- scDRS group level results: {trait}.scdrs_group.celltype
- H5AD file: 11day_DA_neurons.h5ad

We have created them in the previous notebook.
:::

## Table of Contents

* [Set up](#section_1)     
* [Visualise scDRS results](#section_2) 
    * [Overlay scDRS score on UMAP](#section_2_1)
    * [Present group-level results](#section_2_2) 
* [Expression of the ADHD genes in the Jerber dataset](#section_3) 
* [Sensitivity analyses](#section_4)     
    * [Impact of gene set size](#section_4_1)

# 1. Set up <a class="anchor" id="section_1"></a>

In [1]:
#Load the different libraries 
import scdrs
import scanpy as sc
from anndata import AnnData
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings
import math

warnings.filterwarnings("ignore")

# 2. Visualise scDRS results <a class="anchor" id="section_2"></a>

Now that we have calculated the disease score for each cell in the dataset, as well as the celltype level associations, we can present the results in different ways.

## 2.1 Overlay the disease scores over the UMAP representation of the dataset <a class="anchor" id="section_2_1"></a>
The dataset can be represented using UMAP projections. We can then color each cell based on it's disease score. 
<br>
We will first create a plot that shows the UMAP representation where the cells are colored by cell types, and one where the cells are colored by disease score.
                                                                                                                     

In [2]:
gs_adhd_magma

NameError: name 'gs_adhd_magma' is not defined

In [ ]:
#Load the .gs file that contains the top 100 MAGMA genes used for the analysis in the 11day neurons dataset
gs_adhd_magma = pd.read_csv("output/scdrs/munge_adhd_magma_100.gs", sep = '\t', index_col = 0)

#Load the scRNA seq dataset into an AnnData object
adata = sc.read_h5ad("./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad") 

#Load the scores from the scoring step output file and save them in a dictionary
dict_score = {
    trait: pd.read_csv(f"./input/scdrs/script_02/11day_magma100_{trait}.full_score.gz", sep="\t", index_col=0)
    for trait in gs_adhd_magma.index
}

#Add the scores to the adata object
for trait in dict_score:
    adata.obs[trait] = dict_score[trait]["norm_score"]

In [ ]:
#Create the plot with the UMAP colored by cell type
sc.set_figure_params(figsize=[4, 4], dpi=150)
sc.pl.umap(
    adata,
    color="celltype",
    ncols=1,
    color_map="RdBu_r",
)

#Create the plot with the UMAP colored by scDRS score
sc.pl.umap(
    adata,
    color=dict_score.keys(),
    color_map="RdBu_r",
    vmin=-5,
    vmax=5,
    s=20
)

## 2.2 Present cell-type level association results <a class="anchor" id="section_2_2"></a>
There are many ways to present the association with the different cells types, here we will plot the -log10(p-value) as a function of the cell type

In [ ]:
#Load the results of the celltype level analysis 
group_results = pd.read_csv("./input/scdrs/script_02/11day_magma100_ADHD.scdrs_group.celltype", sep = '\t')
group_results.head()

In [ ]:
#Plot the association p-value as a function of the cell type
fig, ax = plt.subplots(figsize=(3.5, 3.5))

sns.scatterplot(
        data=group_results,
        x="group",
        y=-np.log(group_results["assoc_mcp"]),
        label=trait,
        marker="o",
        ax=ax,
    )

#We add a dotted line that represent the lowest value possible for assoc_mcp
ax.hlines(y=-np.log10(0.0009991), xmin = 0, xmax = 2, linestyle = ':', colors = 'black')


ax.set_xlabel("Cell type")
ax.set_ylabel("-log10(assoc_mcp)")
fig.show()

# 3. Expression of ADHD genes in the Jerber dataset <a class="anchor" id="section_3"></a>
So far we have focused on the 11day differenciated neurons from the Jerber dataset, but 30 day and 52 day differenciated neurons are also available. We have computed the scDRS score and the cell type level association with the ADHD gene set for these two additional sets (the results are available in the /output folder)
In this section, you will present the results for the ADHD gene set in the full Jerber dataset

In [ ]:
#Paths to the different files 

#Different Jerber datasets
h5ad_11day = "./reference_data/scDRS_Jerber_dataset/11day_DA_neurons.h5ad"
h5ad_30day = "./reference_data/scDRS_Jerber_dataset/30day_DA_neurons.h5ad"
h5ad_52day = "./reference_data/scDRS_Jerber_dataset/52day_DA_neurons.h5ad"

#Gene set file with top 100 ADHD MAGMA genes
gs_file_adhd = "./output/scdrs/munge_adhd_magma_100.gs"

#scDRS results for 11day neurons
scdrs_adhd_11day = "./input/scdrs/script_02/11day_magma100_ADHD.full_score.gz"
group_adhd_11day = "./input/scdrs/script_02/11day_magma100_ADHD.scdrs_group.celltype"

#scDRS results for 30day neurons
scdrs_adhd_30day = "./input/scdrs/script_02/30day_magma100_ADHD.full_score.gz"
group_adhd_30day = "./input/scdrs/script_02/30day_magma100_ADHD.scdrs_group.celltype"

#scDRS results for 52day neurons
scdrs_adhd_52day = "./input/scdrs/script_02/52day_magma100_ADHD.full_score.gz"
group_adhd_52day = "./input/scdrs/script_02/52day_magma100_ADHD.scdrs_group.celltype"

In [ ]:
### LOAD AND FORMAT INPUT DATA
#Load the .gs file that contains the top 100 MAGMA genes
gs_adhd_magma = pd.read_csv(gs_file_adhd, sep = '\t', index_col = 0)


## 11 day neurons
#Load the scRNA seq datasets into an AnnData objects
adata_11 = sc.read_h5ad(h5ad_11day) 

#Load the scores from the scoring step output file and save them in a dictionary
dict_score_11 = {
    trait: pd.read_csv(scdrs_adhd_11day, sep="\t", index_col=0)
    for trait in gs_adhd_magma.index
}

#Add the scores to the adata object
for trait in dict_score_11:
    adata_11.obs[trait] = dict_score_11[trait]["norm_score"]

#Load the group-level results 
group_results_11 = pd.read_csv(group_adhd_11day, sep = '\t')



## 30 day neurons
#Load the scRNA seq datasets into an AnnData objects
adata_30 = sc.read_h5ad(h5ad_30day) 

#Load the scores from the scoring step output file and save them in a dictionary
dict_score_30 = {
    trait: pd.read_csv(scdrs_adhd_30day, sep="\t", index_col=0)
    for trait in gs_adhd_magma.index
}

#Add the scores to the adata object
for trait in dict_score_30:
    adata_30.obs[trait] = dict_score_30[trait]["norm_score"]

#Load the group-level results 
group_results_30 = pd.read_csv(group_adhd_30day, sep = '\t')



## 52 day neurons
#Load the scRNA seq datasets into an AnnData objects
adata_52 = sc.read_h5ad(h5ad_52day) 

#Load the scores from the scoring step output file and save them in a dictionary
dict_score_52 = {
    trait: pd.read_csv(scdrs_adhd_52day, sep="\t", index_col=0)
    for trait in gs_adhd_magma.index
}

#Add the scores to the adata object
for trait in dict_score_52:
    adata_52.obs[trait] = dict_score_52[trait]["norm_score"]

#Load the group-level results 
group_results_52 = pd.read_csv(group_adhd_52day, sep = '\t')


In [ ]:
### UNIFY COLORING
# Collect all possible celltype categories across datasets -> we want to harmonize the color across datasets
all_categories = sorted(set().union(*[
    ad.obs["celltype"].cat.categories for ad in (adata_11, adata_30, adata_52)
]))

# Create a color palette with the same length
palette = sc.pl.palettes.default_20  # or any palette you prefer

# Map category -> color
color_map = dict(zip(all_categories, palette[:len(all_categories)]))

#Add the new color levels to the three adata objets
for ad in (adata_11, adata_30, adata_52):
    # Ensure same category order
    ad.obs["celltype"] = ad.obs["celltype"].cat.set_categories(all_categories)
    
    # Assign colors in correct order
    ad.uns["celltype_colors"] = [
        color_map[c] for c in ad.obs["celltype"].cat.categories
    ]

### INITIATE FIGURE
fig, axes = plt.subplots(3,3, figsize = (10, 8))
axes = axes.flatten()

### ROW 1: UMAP COLORED BY CELLTYPE
for i, ad in enumerate((adata_11, adata_30, adata_52)):
    # Remove categories not present in this dataset
    ad.obs["celltype"] = ad.obs["celltype"].cat.remove_unused_categories()
    
    # Re-assign colors in the correct (now reduced) order
    ad.uns["celltype_colors"] = [
        color_map[c] for c in ad.obs["celltype"].cat.categories
    ]
    sc.pl.umap(
        ad,
        color="celltype",
        ncols=1,
        color_map="RdBu_r",
        vmin=-5,
        vmax=5,
        ax=axes[i],
        show=False
    )

### ROW 2: UMAP COLORED BY SCDRS SCORE
for i, (ad, keys), in enumerate(zip((adata_11, adata_30, adata_52),(dict_score_11, dict_score_30, dict_score_52))):
    sc.pl.umap(
        ad,
        color=list(keys),
        color_map="RdBu_r",
        vmin=-5,
        vmax=5,
        s=20,
        ax=axes[i+3],
        show=False
    )

### ROW 3: SCATTER PLOT OF ASSOCIATION P-VALUE PER CELLTYPE
for i, df, in enumerate((group_results_11, group_results_30, group_results_52)):
    ax = axes[i + 6]

    sns.scatterplot(
        data=df,
        x="group",
        y=-np.log10(df['assoc_mcp']),
        marker="o",
        ax=ax
    )

    ax.hlines(
        y=-np.log10(0.0009991),
        xmin=0,
        xmax=len(df["group"]) - 1,
        linestyle=":",
        colors="black"
    )

    ax.set_xlabel("Cell type")
    ax.set_ylabel("-log10(assoc_mcp)")
    ax.tick_params(axis='x', rotation=45)

    
plt.tight_layout()
plt.show()

::: callout-note
From the figure created above, answer the following **questions**:

- From the scDRS scores, can you tell if any cell type is associated with ADHD?
- Is this reflected in the group-level analysis? Reminder: the lowest p-value possible is 0.0009991
- Can you comment on the association of ADHD MAGMA genes with the cells in this dataset?
:::

# 4. Sensitivity analyses <a class="anchor" id="section_4"></a>
There are a few things to consider when using scDRS (or other tools) to associate genes with cell types. Two of these considerations are:
- How many genes should you include in your gene set?
- What is the impact of the tool you are using to identify the genes on the downstream analyses?

In this section, you will focus on the first question, and  create a few plots to visualise potential differences

In [ ]:
#Paths to the different files 

#Jerber dataset
h5ad_52day = "./reference_data/scDRS_Jerber_dataset/52day_DA_neurons.h5ad"

#Gene set files 
gs_file_100 = "./input/scdrs/script_02/munge_adhd_magma_100.gs"
gs_file_300 = "./input/scdrs/script_02/munge_adhd_magma_300.gs"
gs_file_500 = "./input/scdrs/script_02/munge_adhd_magma_500.gs"


#scDRS results for top 100 genes
scdrs_adhd_100 = "./input/scdrs/script_02/52day_magma100_ADHD.full_score.gz"
group_adhd_100 = "./input/scdrs/script_02/52day_magma100_ADHD.scdrs_group.celltype"

#scDRS results for top 300 genes
scdrs_adhd_300 = "./input/scdrs/script_02/52day_magma300_ADHD.full_score.gz"
group_adhd_300 = "./input/scdrs/script_02/52day_magma300_ADHD.scdrs_group.celltype"

#scDRS results for top 500 genes
scdrs_adhd_500 = "./input/scdrs/script_02/52day_magma500_ADHD.full_score.gz"
group_adhd_500 = "./input/scdrs/script_02/52day_magma300_ADHD.scdrs_group.celltype"

## 4.1 Impact of gene set size <a class="anchor" id="section_4_1"></a>
We have computed the scDRS scores and cell type association p-values for the Jerber dataset using top 100, 300, and 500 ADHD genes identified by MAGMA. The results available in the input_files/script_02/ folder. 
You will now plot the results for these 3 gene sets and compare them. To simplify, we will focus on the results for the 52day differenciated neurons.

In [ ]:
### LOAD AND FORMAT INPUT DATA

#100 genes
#Load the .gs file that contains the top 100 MAGMA genes
gs_100 = pd.read_csv(gs_file_100, sep = '\t', index_col = 0)

#Load the scores from the scoring step output file and save them in a dictionary
dict_score_100 = {
    trait: pd.read_csv(scdrs_adhd_100, sep="\t", index_col=0)
    for trait in gs_100.index
}

#Add the scores to the adata object
for trait in dict_score_100:
    adata_52.obs['ADHD_100'] = dict_score_100[trait]["norm_score"]
    
#Load the group level results
group_results_100 = pd.read_csv(group_adhd_100, sep = '\t')



#300 genes
#Load the .gs file that contains the top 100 MAGMA genes
gs_300 = pd.read_csv(gs_file_300, sep = '\t', index_col = 0)

#Load the scores from the scoring step output file and save them in a dictionary
dict_score_300 = {
    trait: pd.read_csv(scdrs_adhd_300, sep="\t", index_col=0)
    for trait in gs_300.index
}

#Add the scores to the adata object
for trait in dict_score_300:
    adata_52.obs['ADHD_300'] = dict_score_300[trait]["norm_score"]
    
#Load the group level results
group_results_300 = pd.read_csv(group_adhd_300, sep = '\t')



#500 genes
#Load the .gs file that contains the top 100 MAGMA genes
gs_500 = pd.read_csv(gs_file_500, sep = '\t', index_col = 0)

#Load the scores from the scoring step output file and save them in a dictionary
dict_score_500 = {
    trait: pd.read_csv(scdrs_adhd_500, sep="\t", index_col=0)
    for trait in gs_500.index
}

#Add the scores to the adata object
for trait in dict_score_500:
    adata_52.obs['ADHD_500'] = dict_score_500[trait]["norm_score"]
    
#Load the group level results
group_results_500 = pd.read_csv(group_adhd_500, sep = '\t')


In [ ]:
### INITIATE FIGURE
fig, axes = plt.subplots(3,3, figsize = (10, 8))
axes = axes.flatten()

### ROW 1: UMAP COLORED BY CELLTYPE

sc.pl.umap(
    adata_52,
    color="celltype",
    ncols=1,
    color_map="RdBu_r",
    vmin=-5,
    vmax=5,
    ax=axes[0],
    show=False
 )

### ROW 2: UMAP COLORED BY SCDRS SCORE
for i, keys, in enumerate(("ADHD_100", "ADHD_300", "ADHD_500")):
    sc.pl.umap(
        adata_52,
        color=keys,
        color_map="RdBu_r",
        vmin=-5,
        vmax=5,
        s=20,
        ax=axes[i+3],
        show=False
)

### ROW 3: SCATTER PLOT OF ASSOCIATION P-VALUE PER CELLTYPE
for i, df, in enumerate((group_results_100, group_results_300, group_results_500)):
    ax = axes[i + 6]

    sns.scatterplot(
        data=df,
        x="group",
        y=-np.log10(df['assoc_mcp']),
        marker="o",
        ax=ax
    )

    ax.hlines(
        y=-np.log10(0.0009991),
        xmin=0,
        xmax=len(df["group"]) - 1,
        linestyle=":",
        colors="black"
    )

    ax.set_xlabel("Cell type")
    ax.set_ylabel("-log10(assoc_mcp)")
    ax.tick_params(axis='x', rotation=45)

    
plt.tight_layout()
plt.show()

::: callout-note
From the figure created above, answer the following **questions**:

- Comment on the differences between the different gene set sizes.
- Does gene set size has a strong impact here? Why do we need to consider the size of the gene set? 
:::